# 3 - Vectorization (TF-IDF)

**Converting the cleaned text from `2_preprocessing.ipynb` into numerical TF-IDF matrices ready for the models, and encoding the sentiment labels into numerical form.**

TF-IDF (Term Frequency – Inverse Document Frequency) - Used in text processing to determine how important a word is in a particular document relative to the entire collection of documents.

- TF (Term Frequency) – how often a word appears in a single document
- IDF (Inverse Document Frequency) – how rare a word is across all documents
- TF-IDF = TF × IDF – the final weight of the word

<br>

Steps:

1. Loading the baseline, stemmed, lemmatized, and Lancaster variants (train/test, separately)

2. Encoding labels (`positive`/`negative` → `1`/`0`)

3. Vectorization (converting to numerical TF-IDF matrices) separately for each variant:
    1. Vocabulary diagnostics BEFORE filtering (reference point)
    
    2. TF-IDF vectorization with `min_df=10`, `max_df=0.85`, `ngram_range=(1, 2)` - fit only on train, transform on test - separately for each variant (preventing data leakage — the vocabulary is learned exclusively from the training set so that the model does not "peek" at the test data and unrealistically improve the metrics)
    
    3. Vocabulary diagnostics AFTER filtering (before/after vocabulary, unigrams/bigrams, matrix shape, sparsity, checking popular terms, examples of eliminated terms)

6. Analysis of `min_df=10`, `min_df=15`, and `min_df=20`

7. Decision on the `min_df` value and `max_features` based on actual numbers from the diagnostics

8. Saving matrices (`.npz`), labels (`.npy`), and fitted vectorizers (`.joblib`)

<br>

Note on `min_df`/`max_df` values:
- Initially, values `min_df=5`, `max_df=0.9` were tested, resulting in (150K+ terms in the vocabularies)
- We tightened them to `min_df=10`, `max_df=0.85` - the principle remains the same, only the threshold was adjusted based on the actual diagnostics
- It was verified that "not" appears in 82.4% of reviews in the training set, so it remains below the `max_df=0.85` threshold - the negation is preserved


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

## Loading Processed Data

We load all pairs of files (baseline, stemmed, lemmatized, and Lancaster).


In [2]:
BASELINE_DIR = Path("../data/processed/baseline")
STEMMED_DIR = Path("../data/processed/stemmed")
LEMMATIZED_DIR = Path("../data/processed/lemmatized")
LANCASTER_DIR = Path("../data/processed/lancaster")

base_train_df = pd.read_csv(BASELINE_DIR / "train.csv")
base_test_df = pd.read_csv(BASELINE_DIR / "test.csv")

stem_train_df = pd.read_csv(STEMMED_DIR / "train.csv")
stem_test_df = pd.read_csv(STEMMED_DIR / "test.csv")

lem_train_df = pd.read_csv(LEMMATIZED_DIR / "train.csv")
lem_test_df = pd.read_csv(LEMMATIZED_DIR / "test.csv")

lanc_train_df = pd.read_csv(LANCASTER_DIR / "train.csv")
lanc_test_df = pd.read_csv(LANCASTER_DIR / "test.csv")

print(f"Baseline   - train: {base_train_df.shape[0]}, test: {base_test_df.shape[0]}")
print(f"Stemmed    - train: {stem_train_df.shape[0]}, test: {stem_test_df.shape[0]}")
print(f"Lemmatized - train: {lem_train_df.shape[0]}, test: {lem_test_df.shape[0]}")
print(f"Lancaster  - train: {lanc_train_df.shape[0]}, test: {lanc_test_df.shape[0]}")

Baseline   - train: 39665, test: 9917
Stemmed    - train: 39665, test: 9917
Lemmatized - train: 39665, test: 9917
Lancaster  - train: 39665, test: 9917


## Label Encoding

Sentiment is currently text (`"positive"`/`"negative"`) - sklearn models expect numbers, so we convert it to a binary value: `positive → 1`, `negative → 0`.

This encoding is consistent across all models and enables standard metrics (e.g. ROC-AUC) that require a numerical class.


In [3]:
def encode_labels(sentiment_series: pd.Series) -> np.ndarray:
    return (sentiment_series == "positive").astype(int).to_numpy()

y_train_base = encode_labels(base_train_df["sentiment"])
y_test_base = encode_labels(base_test_df["sentiment"])

y_train_stem = encode_labels(stem_train_df["sentiment"])
y_test_stem = encode_labels(stem_test_df["sentiment"])

y_train_lem = encode_labels(lem_train_df["sentiment"])
y_test_lem = encode_labels(lem_test_df["sentiment"])

y_train_lanc = encode_labels(lanc_train_df["sentiment"])
y_test_lanc = encode_labels(lanc_test_df["sentiment"])

## Vectorization and Diagnostics (Shared Function for All Variants)

`min_df` and `max_df` are not default values, but experimental values that need to be confirmed using actual numbers. Therefore, we first build an unfiltered vocabulary (just to see the "raw" state), and then compare it with the filtered one.

We use `min_df=10` - it eliminates terms that appear in fewer than ten reviews, thereby reducing the influence of extremely rare terms, including potential typographical errors and random n-gram patterns.

We use `max_df=0.85` - we set it as a corpus-specific filter (terms that appear in more than 85% of reviews), and then explicitly check which terms it actually removed (it is important that negation words are not eliminated because we want them to remain as stated in `2_preprocessing.ipynb`).

This function is called four times below (baseline, stemmed, lemmatized, and Lancaster variants) - it is defined once here to avoid code duplication.

We check the vocabulary both before and after filtering for data comparison.

<br>

The output gives us insight into:

- Number of tokens in the vocabulary before and after filtering, for unigrams and bigrams

- Matrix shape (`number_of_reviews × number_of_terms`) - each row is one review, each column is one term from the vocabulary. The value in cell `[x, y]` is the TF-IDF weight of term `y` in review `x` - this is the form in which the data enters the model

- Non-zero elements - the number of cells that are not zero, i.e. how many terms actually appear somewhere in the corpus (summed across all reviews). Since each individual review uses only a small portion of the entire vocabulary, this number is much smaller than the total number of cells in the matrix

- Sparsity = `1 - (non_zero_elements / (number_of_reviews × number_of_terms))` - the percentage of cells that ARE zero. A value close to 100% is expected, which is why we use the `scipy.sparse` format, which stores only non-zero values instead of the entire matrix

- Checking some popular words and negation bigrams (bigrams usually have a lower frequency of occurrence than the individual words they are composed of)

- Examples of eliminated terms, separated by cause (`min_df` vs `max_df`)


In [4]:
POPULAR_TERMS = ["movi", "movie", "film", "actor", "good", "bad", "not", "no", "nor", "never", "neither"]

NEGATION_BIGRAMS = ["not good", "not bad", "not great", "not enjoy", "not recommend", "no good", "never good",]


def analyze_variant(train_texts: pd.Series, test_texts: pd.Series, variant_name: str):
    # "Sirov" (nefiltriran) recnik - da vidimo stanje pre min_df/max_df filtriranja.
    # Zanima nas samo da li se termin pojavljuje u recenziji, ne koliko puta - zato CountVectorizer(binary=True), ne jeftinije za ovu proveru od TfidfVectorizer.
    raw_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=1, max_df=1.0, binary=True)
    X_raw = raw_vectorizer.fit_transform(train_texts)
    raw_vocab_size = len(raw_vectorizer.vocabulary_)
    n_docs = X_raw.shape[0]

    # Stvarni TF-IDF vectorizer, sa filtriranjem - FIT samo na train, TRANSFORM na test
    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=10, max_df=0.85)
    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    vocab = vectorizer.vocabulary_
    vocab_size = len(vocab)
    n_unigrams = sum(1 for term in vocab if " " not in term)
    n_bigrams = vocab_size - n_unigrams

    # Ispis statistike i provera raznih termina
    print(f"========== {variant_name} ==========")
    print(f"Train reviews: {X_train.shape[0]}")
    print(f"Test reviews:  {X_test.shape[0]}")
    print(f"\nVocabulary BEFORE filtering (min_df=1, max_df=1.0): {raw_vocab_size}")
    print(f"Vocabulary AFTER filtering (min_df=10, max_df=0.85): {vocab_size}")
    print(f"\nEliminated: {raw_vocab_size - vocab_size} terms ({(raw_vocab_size - vocab_size) / raw_vocab_size:.1%})")
    print(f"  - unigrams in final vocabulary: {n_unigrams}")
    print(f"  - bigrams in final vocabulary:  {n_bigrams}")
    print(f"\nTrain matrix shape: {X_train.shape}")
    print(f"Number of non-zero elements: {X_train.nnz:,}")
    sparsity = 1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1])
    print(f"Sparsity: {sparsity:.2%}")

    print(f"\nChecking popular terms:")
    for term in POPULAR_TERMS:
        if term not in raw_vectorizer.vocabulary_:
            continue
        if term in vocab:
            idx = vocab[term]
            doc_count = (X_train[:, idx] > 0).sum()
            print(f"  '{term}': KEPT, appears in {doc_count} reviews ({doc_count/n_docs:.2%})")
        else:
            print(f"  '{term}': NOT in vocabulary")

    print(f"\nChecking negation bigrams:")
    for phrase in NEGATION_BIGRAMS:
        if phrase in vocab:
            idx = vocab[phrase]
            doc_count = (X_train[:, idx] > 0).sum()
            print(f"  '{phrase}': KEPT, appears in {doc_count} reviews ({doc_count/n_docs:.2%})")
        else:
            print(f"  '{phrase}': NOT in vocabulary")

    # Primeri eliminisanih termina, razdvojeni po uzroku (min_df vs max_df)
    doc_freq_counts = np.asarray(X_raw.sum(axis=0)).flatten()

    eliminated = set(raw_vectorizer.vocabulary_.keys()) - set(vocab.keys())
    low_freq, high_freq = [], []
    for term in eliminated:
        idx = raw_vectorizer.vocabulary_[term]
        df_count = doc_freq_counts[idx]
        if df_count < 10:
            low_freq.append((term, int(df_count)))
        else:
            high_freq.append((term, df_count / n_docs))

    low_freq.sort(key=lambda x: -x[1])
    high_freq.sort(key=lambda x: -x[1])
    print(f"\nEliminated due to min_df (too few reviews): {len(low_freq)}")
    print(f"  Examples (term, review count): {low_freq[:8]}")
    print(f"\nEliminated due to max_df (too many reviews): {len(high_freq)}")
    print(f"  Examples (term, % of reviews): {[(t, f'{p:.1%}') for t, p in high_freq[:8]]}")

    return vectorizer, X_train, X_test

## Applying to the baseline variant


In [5]:
vectorizer_base, X_train_base, X_test_base = analyze_variant(base_train_df["review_baseline"], base_test_df["review_baseline"], "BASELINE VARIANT")

========== BASELINE VARIANT ==========
Train reviews: 39665
Test reviews:  9917

Vocabulary BEFORE filtering (min_df=1, max_df=1.0): 2642943
Vocabulary AFTER filtering (min_df=10, max_df=0.85): 64803

Eliminated: 2578140 terms (97.5%)
  - unigrams in final vocabulary: 22855
  - bigrams in final vocabulary:  41948

Train matrix shape: (39665, 64803)
Number of non-zero elements: 5,011,176
Sparsity: 99.81%

Checking popular terms:
  'movie': KEPT, appears in 24264 reviews (61.17%)
  'film': KEPT, appears in 22002 reviews (55.47%)
  'actor': KEPT, appears in 3024 reviews (7.62%)
  'good': KEPT, appears in 15048 reviews (37.94%)
  'bad': KEPT, appears in 9368 reviews (23.62%)
  'not': KEPT, appears in 32694 reviews (82.43%)
  'no': KEPT, appears in 12913 reviews (32.56%)
  'nor': KEPT, appears in 931 reviews (2.35%)
  'never': KEPT, appears in 7995 reviews (20.16%)
  'neither': KEPT, appears in 783 reviews (1.97%)

Checking negation bigrams:
  'not good': KEPT, appears in 1313 reviews (3.31

## Applying to the stemmed variant


In [6]:
vectorizer_stem, X_train_stem, X_test_stem = analyze_variant(stem_train_df["review_stemmed"], stem_test_df["review_stemmed"], "STEMMED VARIANT")

========== STEMMED VARIANT ==========
Train reviews: 39665
Test reviews:  9917

Vocabulary BEFORE filtering (min_df=1, max_df=1.0): 2201319
Vocabulary AFTER filtering (min_df=10, max_df=0.85): 66682

Eliminated: 2134637 terms (97.0%)
  - unigrams in final vocabulary: 15566
  - bigrams in final vocabulary:  51116

Train matrix shape: (39665, 66682)
Number of non-zero elements: 5,240,315
Sparsity: 99.80%

Checking popular terms:
  'movi': KEPT, appears in 25714 reviews (64.83%)
  'movie': NOT in vocabulary
  'film': KEPT, appears in 23592 reviews (59.48%)
  'actor': KEPT, appears in 8025 reviews (20.23%)
  'good': KEPT, appears in 15230 reviews (38.40%)
  'bad': KEPT, appears in 9911 reviews (24.99%)
  'not': KEPT, appears in 32697 reviews (82.43%)
  'no': KEPT, appears in 12913 reviews (32.56%)
  'nor': KEPT, appears in 931 reviews (2.35%)
  'never': KEPT, appears in 7995 reviews (20.16%)
  'neither': KEPT, appears in 783 reviews (1.97%)

Checking negation bigrams:
  'not good': KEPT, a

## Applying to the lemmatized variant


In [7]:
vectorizer_lem, X_train_lem, X_test_lem = analyze_variant(lem_train_df["review_lemmatized"], lem_test_df["review_lemmatized"], "LEMMATIZED VARIANT")

========== LEMMATIZED VARIANT ==========
Train reviews: 39665
Test reviews:  9917

Vocabulary BEFORE filtering (min_df=1, max_df=1.0): 2318157
Vocabulary AFTER filtering (min_df=10, max_df=0.85): 66834

Eliminated: 2251323 terms (97.1%)
  - unigrams in final vocabulary: 18751
  - bigrams in final vocabulary:  48083

Train matrix shape: (39665, 66834)
Number of non-zero elements: 5,165,555
Sparsity: 99.81%

Checking popular terms:
  'movie': KEPT, appears in 25713 reviews (64.83%)
  'film': KEPT, appears in 23528 reviews (59.32%)
  'actor': KEPT, appears in 8014 reviews (20.20%)
  'good': KEPT, appears in 16617 reviews (41.89%)
  'bad': KEPT, appears in 11677 reviews (29.44%)
  'not': KEPT, appears in 32694 reviews (82.43%)
  'no': KEPT, appears in 12916 reviews (32.56%)
  'nor': KEPT, appears in 931 reviews (2.35%)
  'never': KEPT, appears in 7995 reviews (20.16%)
  'neither': KEPT, appears in 783 reviews (1.97%)

Checking negation bigrams:
  'not good': KEPT, appears in 1316 reviews (

## Applying to the Lancaster variant


In [8]:
vectorizer_lanc, X_train_lanc, X_test_lanc = analyze_variant(lanc_train_df["review_lancaster"], lanc_test_df["review_lancaster"], "LANCASTER VARIANT")

========== LANCASTER VARIANT ==========
Train reviews: 39665
Test reviews:  9917

Vocabulary BEFORE filtering (min_df=1, max_df=1.0): 2026434
Vocabulary AFTER filtering (min_df=10, max_df=0.85): 68378

Eliminated: 1958056 terms (96.6%)
  - unigrams in final vocabulary: 13094
  - bigrams in final vocabulary:  55284

Train matrix shape: (39665, 68378)
Number of non-zero elements: 5,342,051
Sparsity: 99.80%

Checking popular terms:
  'film': KEPT, appears in 23596 reviews (59.49%)
  'good': KEPT, appears in 15240 reviews (38.42%)
  'bad': KEPT, appears in 9917 reviews (25.00%)
  'not': KEPT, appears in 33058 reviews (83.34%)
  'no': KEPT, appears in 12913 reviews (32.56%)
  'nor': KEPT, appears in 1320 reviews (3.33%)

Checking negation bigrams:
  'not good': KEPT, appears in 1323 reviews (3.34%)
  'not bad': KEPT, appears in 1392 reviews (3.51%)
  'not great': NOT in vocabulary
  'not enjoy': KEPT, appears in 214 reviews (0.54%)
  'not recommend': KEPT, appears in 524 reviews (1.32%)
  '

## Analysis of `min_df=10`, `min_df=15`, and `min_df=20`

We examine whether increasing `min_df` from 10 to 15 or 20 would unjustifiably remove terms that may contain useful signal for sentiment classification.

We compare three values:
- `min_df=10` — retains all terms that appear in at least 10 reviews and serves as the basis for comparison.
- `min_df=15` — removes terms that appear in 10–14 reviews.
- `min_df=20` — removes terms that appear in 10–19 reviews.

### Impact on Feature Space Size

For each `min_df` value, we examine:

- Vocabulary — the total number of terms that remain available to the model.
- Non-zero elements — the number of non-zero values in the matrix. The TF-IDF matrix is highly sparse, with most cells being zero, so reducing the number of terms does not necessarily reduce the number of non-zero elements proportionally.

(Percentages in parentheses show the reduction relative to `min_df=10`.)

### Do the Terms We Remove Carry Sentiment Signal?

The number of occurrences of a term alone is not enough to determine whether the term is irrelevant. A term that appears in only 10–14 reviews may be rare, but if it appears in 90% of positive and only 10% of negative reviews in which it occurs, it carries potentially very strong sentiment signal.

Therefore, for terms that would be removed with `min_df=15` or `min_df=20`, we check their distribution between positive and negative reviews.

Terms are grouped according to the percentage of reviews of the same class in which they appear:
- 90–100% — very strong class signal
- 80–90% — strong class signal
- 70–80% — moderate class signal
- 60–70% — weaker class signal
- 50–60% — approximately evenly distributed between classes

### Average Deviation from 50/50

Finally, we calculate the average deviation from the ideally even `50/50` distribution.

(A value of `0%` means that the terms are, on average, equally represented in positive and negative reviews, while a higher value means that they are, on average, more concentrated in one class.)

<br>

We perform this check only on the stemmed and lemmatized data to see the general state; there is no need to perform it on all 4 types of data.


In [9]:
def compare_min_df_signal(train_texts: pd.Series, y_train: np.ndarray, variant_name: str):
    # min_df = 10
    vectorizer_10 = CountVectorizer(ngram_range=(1, 2), min_df=10, max_df=0.85, binary=True)
    X_10 = vectorizer_10.fit_transform(train_texts)
    vocab_10 = vectorizer_10.vocabulary_

    # min_df = 15
    vectorizer_15 = CountVectorizer(ngram_range=(1, 2), min_df=15, max_df=0.85, binary=True)
    X_15 = vectorizer_15.fit_transform(train_texts)
    vocab_15 = vectorizer_15.vocabulary_

    # min_df = 20
    vectorizer_20 = CountVectorizer(ngram_range=(1, 2), min_df=20, max_df=0.85, binary=True)
    X_20 = vectorizer_20.fit_transform(train_texts)
    vocab_20 = vectorizer_20.vocabulary_

    # Osnovna statistika
    vocab_size_10 = len(vocab_10)
    vocab_size_15 = len(vocab_15)
    vocab_size_20 = len(vocab_20)

    nnz_10 = X_10.nnz
    nnz_15 = X_15.nnz
    nnz_20 = X_20.nnz

    # Procenat smanjenja vokabulara u odnosu na min_df=10
    vocab_reduction_15 = 1 - (vocab_size_15 / vocab_size_10)
    vocab_reduction_20 = 1 - (vocab_size_20 / vocab_size_10)

    # Procenat smanjenja non-zero elemenata
    nnz_reduction_15 = 1 - (nnz_15 / nnz_10)
    nnz_reduction_20 = 1 - (nnz_20 / nnz_10)

    # Koliko dodatnih termina se izbacuje
    removed_15 = vocab_size_10 - vocab_size_15
    removed_20 = vocab_size_10 - vocab_size_20

    # Prikaz - broj termina i koliko bi se dodatno izbacilo sa min_df=15 i min_df=20
    comparison_df = pd.DataFrame({
        "min_df=10": [f"{vocab_size_10:,}", f"{nnz_10:,}"],
        "min_df=15": [f"{vocab_size_15:,} (-{vocab_reduction_15:.1%})", f"{nnz_15:,} (-{nnz_reduction_15:.1%})"],
        "min_df=20": [f"{vocab_size_20:,} (-{vocab_reduction_20:.1%})", f"{nnz_20:,} (-{nnz_reduction_20:.1%})"]
    }, index=["Vocabulary", "Non-zero elements"])
    
    print()
    print("=" * 90)
    print(variant_name)
    print("=" * 90)
    print()
    print(comparison_df.to_string())
    print()
    print(f"min_df=15 would additionally remove: {removed_15:,} tokens")
    print(f"min_df=20 would additionally remove: {removed_20:,} tokens")

    # Gradimo vocabulary sa min_df=10
    # Osnova za poređenje - sve što postoji sa min_df=15 ili 20 mora prvo da postoji u ovom vocabularu
    feature_names = vectorizer_10.get_feature_names_out()
    doc_freq = np.asarray(X_10.sum(axis=0)).ravel().astype(int)         # Document frequency = broj recenzija u kojima se termin pojavljuje
    positive_counts = np.asarray(X_10.T @ y_train).ravel().astype(int)  # Broj pozitivnih recenzija za svaki termin    
    negative_counts = doc_freq - positive_counts                        # Broj negativnih recenzija
    positive_pct = positive_counts / doc_freq                           # Procenat pozitivnih među recenzijama u kojima se termin pojavljuje
    skew = np.abs(positive_pct - 0.5)                                   # Koliko je termin udaljen od 50/50

    # DataFrame sa svim potrebnim podacima
    stats = pd.DataFrame({
        "term": feature_names,
        "document_frequency": doc_freq,
        "positive_reviews": positive_counts,
        "negative_reviews": negative_counts,
        "positive_pct": positive_pct,
        "skew": skew
    })

    # Statistika koliko bi se termina izbacilo po tome koliki procenat recenzija je u istom tipu (pozitivne ili negativne)
    removed_for_15 = stats[(stats["document_frequency"] >= 10) & (stats["document_frequency"] < 15)].copy()
    removed_for_20 = stats[(stats["document_frequency"] >= 10) & (stats["document_frequency"] < 20)].copy()

    # Funkcija za brojanje termina po kategorijama
    def count_signal_categories(dataframe):
        pct = dataframe["positive_pct"]
        same_class_pct = np.maximum(pct, 1 - pct)
        categories = {
            "90-100%": (same_class_pct >= 0.90),
            "80-90%": ((same_class_pct >= 0.80) & (same_class_pct < 0.90)),
            "70-80%": ((same_class_pct >= 0.70) & (same_class_pct < 0.80)),
            "60-70%": ((same_class_pct >= 0.60) & (same_class_pct < 0.70)),
            "50-60%": ((same_class_pct >= 0.50) & (same_class_pct < 0.60))
        }
        return {
            category: int(mask.sum())
            for category, mask in categories.items()
        }

    categories_15 = count_signal_categories(removed_for_15)
    categories_20 = count_signal_categories(removed_for_20)
    
    # Prikaz - statistika koliko bi se termina izbacilo vise nego u min_df=10 po tome koliki procenat recenzija je u istom tipu (pozitivne ili negativne)
    print("-" * 90)
    print("How many tokens would be removed by higher min_df, by their positive/negative class distribution (compared to min_df=10)")
    print("-" * 90)

    print(f"90-100% in the same review class: min_df=15 would remove {categories_15['90-100%']:,} tokens | min_df=20 would remove {categories_20['90-100%']:,} tokens")
    print(f"80-90% in the same review class: min_df=15 would remove {categories_15['80-90%']:,} tokens | min_df=20 would remove {categories_20['80-90%']:,} tokens")
    print(f"70-80% in the same review class: min_df=15 would remove {categories_15['70-80%']:,} tokens | min_df=20 would remove {categories_20['70-80%']:,} tokens")
    print(f"60-70% in the same review class: min_df=15 would remove {categories_15['60-70%']:,} tokens | min_df=20 would remove {categories_20['60-70%']:,} tokens")
    print(f"50-60% in the same review class: min_df=15 would remove {categories_15['50-60%']:,} tokens | min_df=20 would remove {categories_20['50-60%']:,} tokens")

    # Prikaz - prosečno odstupanje od 50/50
    print()
    print("-" * 90)
    print("Average deviation from 50/50")
    print("-" * 90)

    mean_skew_15 = (removed_for_15["skew"].mean())
    mean_skew_20 = (removed_for_20["skew"].mean())

    print(f"Tokens with DF 10-14 (that would be removed by min_df=15) : {mean_skew_15:.2%}")
    print(f"Tokens with DF 10-19 (that would be removed by min_df=20) : {mean_skew_20:.2%}")


# STEMMED/LEMMATIZED pozivi
compare_min_df_signal(train_texts=stem_train_df["review_stemmed"], y_train=y_train_stem, variant_name="STEMMED")
compare_min_df_signal(train_texts=lem_train_df["review_lemmatized"], y_train=y_train_lem, variant_name="LEMMATIZED")


STEMMED

                   min_df=10          min_df=15          min_df=20
Vocabulary            66,682    42,906 (-35.7%)    31,521 (-52.7%)
Non-zero elements  5,240,315  4,963,322 (-5.3%)  4,772,415 (-8.9%)

min_df=15 would additionally remove: 23,776 tokens
min_df=20 would additionally remove: 35,161 tokens
------------------------------------------------------------------------------------------
How many tokens would be removed by higher min_df, by their positive/negative class distribution (compared to min_df=10)
------------------------------------------------------------------------------------------
90-100% in the same review class: min_df=15 would remove 2,715 tokens | min_df=20 would remove 3,315 tokens
80-90% in the same review class: min_df=15 would remove 2,878 tokens | min_df=20 would remove 4,409 tokens
70-80% in the same review class: min_df=15 would remove 4,886 tokens | min_df=20 would remove 6,882 tokens
60-70% in the same review class: min_df=15 would remove 6,611

## Decision on `min_df` and Introducing `max_features`

### `min_df` Conclusion

The results show that `min_df=10` is the most reasonable choice.

Increasing `min_df` significantly reduces the vocabulary:
- `min_df=15`: ~35% fewer terms
- `min_df=20`: ~52% fewer terms

However, the number of non-zero elements decreases much less:
- `min_df=15`: ~5%
- `min_df=20`: ~9%

A higher `min_df` removes thousands of potentially useful terms.

With `min_df=15`, more than 2,600 terms in both variants appear in 90–100% of reviews of the same class, which indicates a strong class signal.

This does not mean that all terms are genuine signals, but there are enough of them that they cannot be ignored (mathematically, for a set of 23,776 terms with df 10–14, we would expect approximately ~167 of them to fall into the 90–100% same-class category purely by chance).

<br>

### Introducing `max_features`

The vocabulary after `min_df`/`max_df` filtering contains:
- 64,803 terms without normalization (baseline)
- 66,682 stemmed
- 66,834 lemmatized
- 68,378 Lancaster stemmed

Matrix sparsity is ~99.80%.

These sizes are not computationally problematic for the planned models - therefore, we do not limit the vocabulary with `max_features`.

All 6 models are compared on the same feature space, which is a prerequisite for a fair comparison of algorithms (the difference in results comes exclusively from the models, not from a different input space).

If an individual model (e.g. KNN) has performance issues in this space, we address it locally in its notebook (e.g. `TruncatedSVD` only for KNN).


## Saving the Results

We save:
- Sparse TF-IDF matrices as `.npz` (`Compressed NumPy Archive` is optimized for storing large sparse matrices)
- Numerical labels as `.npy` (a binary format for efficiently storing an array of numbers)
- Fitted vectorizer objects as `.joblib` ("template", i.e. the rules according to which words are converted into numbers, needed later, e.g. for demo prediction on a new review).


In [10]:
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# TF-IDF matrice
sparse.save_npz(BASELINE_DIR / "X_train.npz", X_train_base)
sparse.save_npz(BASELINE_DIR / "X_test.npz", X_test_base)

sparse.save_npz(STEMMED_DIR / "X_train.npz", X_train_stem)
sparse.save_npz(STEMMED_DIR / "X_test.npz", X_test_stem)

sparse.save_npz(LEMMATIZED_DIR / "X_train.npz", X_train_lem)
sparse.save_npz(LEMMATIZED_DIR / "X_test.npz", X_test_lem)

sparse.save_npz(LANCASTER_DIR / "X_train.npz", X_train_lanc)
sparse.save_npz(LANCASTER_DIR / "X_test.npz", X_test_lanc)

# Labele
np.save(BASELINE_DIR / "y_train.npy", y_train_base)
np.save(BASELINE_DIR / "y_test.npy", y_test_base)

np.save(STEMMED_DIR / "y_train.npy", y_train_stem)
np.save(STEMMED_DIR / "y_test.npy", y_test_stem)

np.save(LEMMATIZED_DIR / "y_train.npy", y_train_lem)
np.save(LEMMATIZED_DIR / "y_test.npy", y_test_lem)

np.save(LANCASTER_DIR / "y_train.npy", y_train_lanc)
np.save(LANCASTER_DIR / "y_test.npy", y_test_lanc)

# Fitovani vectorizer-i
joblib.dump(vectorizer_base, MODELS_DIR / "tfidf_vectorizer_base.joblib")
joblib.dump(vectorizer_stem, MODELS_DIR / "tfidf_vectorizer_stem.joblib")
joblib.dump(vectorizer_lem, MODELS_DIR / "tfidf_vectorizer_lem.joblib")
joblib.dump(vectorizer_lanc, MODELS_DIR / "tfidf_vectorizer_lanc.joblib")

print("Saved:")
print(f"  {BASELINE_DIR / 'X_train.npz'}  shape={X_train_base.shape}")
print(f"  {BASELINE_DIR / 'X_test.npz'}   shape={X_test_base.shape}")
print(f"  {STEMMED_DIR / 'X_train.npz'}  shape={X_train_stem.shape}")
print(f"  {STEMMED_DIR / 'X_test.npz'}   shape={X_test_stem.shape}")
print(f"  {LEMMATIZED_DIR / 'X_train.npz'}  shape={X_train_lem.shape}")
print(f"  {LEMMATIZED_DIR / 'X_test.npz'}   shape={X_test_lem.shape}")
print(f"  {LANCASTER_DIR / 'X_train.npz'}  shape={X_train_lanc.shape}")
print(f"  {LANCASTER_DIR / 'X_test.npz'}   shape={X_test_lanc.shape}")

print(f"  {MODELS_DIR / 'tfidf_vectorizer_base.joblib'}")
print(f"  {MODELS_DIR / 'tfidf_vectorizer_stem.joblib'}")
print(f"  {MODELS_DIR / 'tfidf_vectorizer_lem.joblib'}")
print(f"  {MODELS_DIR / 'tfidf_vectorizer_lanc.joblib'}")

Saved:
  ..\data\processed\baseline\X_train.npz  shape=(39665, 64803)
  ..\data\processed\baseline\X_test.npz   shape=(9917, 64803)
  ..\data\processed\stemmed\X_train.npz  shape=(39665, 66682)
  ..\data\processed\stemmed\X_test.npz   shape=(9917, 66682)
  ..\data\processed\lemmatized\X_train.npz  shape=(39665, 66834)
  ..\data\processed\lemmatized\X_test.npz   shape=(9917, 66834)
  ..\data\processed\lancaster\X_train.npz  shape=(39665, 68378)
  ..\data\processed\lancaster\X_test.npz   shape=(9917, 68378)
  ..\models\tfidf_vectorizer_base.joblib
  ..\models\tfidf_vectorizer_stem.joblib
  ..\models\tfidf_vectorizer_lem.joblib
  ..\models\tfidf_vectorizer_lanc.joblib


## Vectorization Conclusions

- Baseline variant: vocabulary before filtering 2,642,943, after 64,803 (22,855 unigrams, 41,948 bigrams, sparsity 99.81%)

- Stemmed variant: vocabulary before filtering 2,201,319, after 66,682 (15,566 unigrams, 51,116 bigrams, sparsity 99.80%)

- Lemmatized variant: vocabulary before filtering 2,318,157, after 66,834 (18,751 unigrams, 48,083 bigrams, sparsity 99.81%)

- Lancaster stemmed variant: vocabulary before filtering 2,026,434, after 68,378 (13,094 unigrams, 55,284 bigrams, sparsity 99.80%)

- Checking key terms and negations:
    - Common words such as "movie" and "film" were retained and were not eliminated by the upper threshold (`max_df=0.85`)
    - The word "not" appeared in 82.4% of reviews, which is close to the threshold, but remained because it is below the 85% limit
    - Important negation bigrams (such as not good, not bad) were retained and were not eliminated by the lower threshold (`min_df=10`)

- Saved matrices and labels: TF-IDF matrices (`X_test.npz` and `X_train.npz`) and numerical labels (`y_test.npy` and `y_train.npy`) were successfully saved in the corresponding folders (`data/processed/stemmed/` and `data/processed/lemmatized/`)

- Saved models: Fitted vectorizer objects were successfully saved in the `models/` folder for later use during prediction on new data
